# 현재 좌석 연결과 핵심 상권 제외 원인 감사
고정 KOPIS 스냅샷과 해당 원문 XML을 읽어 **공연의 개별 공연장 ID와 현재 좌석 수**를 연결할 수 있는지 확인한다. 시설 전체 좌석을 개별 공연장 좌석으로 쓰지 않고, 시설명 추론을 공식 ID 일치로 인정하지 않는다. 현재 좌석의 연결 가능성과 2023~2025년 당시 좌석·관객의 확인은 별개다.

둘째로 대학로(혜화역)·홍대입구역(홍대)의 후보 ID×분기 제외 사유와 우선 검증 목록을 작성한다. 일정 파서를 바꾸거나 재수집하거나 회귀를 추가하지 않는다. 네트워크 요청과 API URL·인증정보 출력은 없다.

## 1. 고정 입력과 원문 명세 확인
`jupyter-notebook/20261006/metadata/kopis_analysis_sources.json`의 검증된 CSV와 raw manifest만 사용한다. 원문 파일은 명세의 SHA256과 같을 때만 읽는다. 원본과 final 파일, 앞선 jupyter-notebook/sandbox/01 결과는 해시로 보호한다.

In [ ]:
# 현재 구조 및 과거 고정 명세의 경로를 함께 해석합니다.
from pathlib import Path as _ProjectPath
import json as _bootstrap_json
_project_home = next(p for p in [_ProjectPath.cwd(), *_ProjectPath.cwd().parents]
                     if (p / 'environment/project_paths.ipynb').is_file())
_project_notebook = _bootstrap_json.loads((_project_home / 'environment/project_paths.ipynb').read_text(encoding='utf-8'))
for _project_cell in _project_notebook['cells']:
    if 'project-runtime' in _project_cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_project_cell['source']), 'environment/project_paths.ipynb', 'exec'), globals())

from pathlib import Path
import json,hashlib,re,xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
from IPython.display import display,Markdown
ROOT=PROJECT_ROOT
REPORTS=project_path('jupyter-notebook/sandbox/reports');DATA=project_path('jupyter-notebook/sandbox/data')
REPORTS.mkdir(parents=True,exist_ok=True);DATA.mkdir(parents=True,exist_ok=True)
sha=lambda path:hashlib.sha256(path.read_bytes()).hexdigest()
inputs={}
def register(relative,expected=None):
    path=project_path(relative);digest=sha(path)
    if expected is not None:assert digest==expected,relative
    if relative not in inputs:inputs[relative]={'path':relative,'sha256_before':digest,'expected_sha256':expected}
    return path
lab=project_json_loads(register('jupyter-notebook/sandbox/metadata/lab_plan.json').read_text(encoding='utf-8'))
for relative,digest in lab['protected_sha256'].items():register(relative,digest)
for relative in ['jupyter-notebook/sandbox/notebooks/01_측정표본_탐색.ipynb','jupyter-notebook/sandbox/reports/01_summary.json','jupyter-notebook/sandbox/reports/01_specification_results.csv']:register(relative)
manifest=project_json_loads(register('jupyter-notebook/20261006/metadata/kopis_analysis_sources.json').read_text(encoding='utf-8'))
assert manifest['validated']
frames={}
for role in ['performance_details','facility_details']:
    spec=manifest['files'][role]
    frames[role]=pd.read_csv(register(spec['path'],spec['sha256']),dtype=str,keep_default_na=False)
performances,facilities=frames['performance_details'],frames['facility_details']
raw_spec=manifest['raw_sources']
raw=pd.read_csv(register(raw_spec['path'],raw_spec['sha256']),dtype=str,keep_default_na=False)
s12=project_json_loads(register('reports/20261006/revised_quarterly/12_dataset_summary.json').read_text(encoding='utf-8'))
assert s12['status']=='complete' and s12['all_inputs_unchanged']
fixed={}
for name in ['performance_quarter_schedule.csv','performance_schedule_audit.csv','facility_area_links.csv','area_quarter_exposure.csv','area_centroids.csv','revised_quarterly_panel.csv']:
    relative='dataset/analysis/final/revised_quarterly/'+name
    fixed[name]=pd.read_csv(register(relative,s12['outputs'][relative]['sha256']),dtype={'area_id':str,'mt20id':str,'mt10id':str,'quarter':str,'industry_id':str},float_precision='round_trip')
pq=fixed['performance_quarter_schedule.csv'];audit=fixed['performance_schedule_audit.csv'];links=fixed['facility_area_links.csv']
exposure=fixed['area_quarter_exposure.csv'];areas=fixed['area_centroids.csv'];panel=fixed['revised_quarterly_panel.csv']
candidate_ids=set(pq.mt20id);candidate_facilities=set(pq.mt10id)
assert len(candidate_ids)==12839 and len(pq)==14202

## 2. 시설 CSV에서 빠진 개별 공연장 배열
시설 CSV의 `seatscale`과 XML `mt13s/mt13/seatscale`를 구분한다. XML의 개별 공연장에는 `mt13id`가 있으며 공연 CSV의 `mt13id`와 공식 ID로 대조할 수 있다. 쉼표가 있는 정수만 수치로 해석하며 0·빈 값·비정형 값은 양의 좌석 가용으로 처리하지 않는다.

In [ ]:
def parse_seat_integer(value):
    text=str(value or '').strip()
    if not text:return np.nan,'missing'
    if not re.fullmatch(r'(?:\d+|\d{1,3}(?:,\d{3})+)',text):return np.nan,'not_integer_text'
    value=int(text.replace(',',''))
    return value,'positive_integer' if value>0 else 'zero_or_nonpositive'

assert parse_seat_integer('1,004')==(1004,'positive_integer')
assert parse_seat_integer('0')[1]=='zero_or_nonpositive'
assert parse_seat_integer('1,2')[1]=='not_integer_text'
relevant_facilities=set(performances.mt10id)
raw_facility=raw.loc[raw.table.eq('facility_details')&raw.record_id.isin(relevant_facilities)]
assert raw_facility.record_id.nunique()==len(relevant_facilities)
hall_rows=[];facility_rows=[];hall_tags=set()
for row in raw_facility.itertuples(index=False):
    path=register(row.path,row.sha256)
    assert path.resolve().is_relative_to((project_path('dataset/raw')).resolve())
    root=ET.fromstring(path.read_bytes());db=root if root.tag=='db' else root.find('db')
    assert db is not None and db.findtext('mt10id')==row.record_id
    halls=db.findall('./mt13s/mt13')
    capacity,cap_status=parse_seat_integer(db.findtext('seatscale'))
    declared,declared_status=parse_seat_integer(db.findtext('mt13cnt'))
    facility_rows.append(dict(mt10id=row.record_id,fcltynm=db.findtext('fcltynm'),
        facility_seatscale_raw=db.findtext('seatscale'),facility_seats=capacity,facility_seat_status=cap_status,
        declared_hall_count=declared,xml_hall_count=len(halls),raw_source_path=row.path,raw_source_sha256=row.sha256))
    for position,hall in enumerate(halls):
        hall_tags.update(e.tag for e in hall)
        seats,status=parse_seat_integer(hall.findtext('seatscale'))
        hall_rows.append(dict(mt10id=row.record_id,mt13id=(hall.findtext('mt13id') or '').strip(),
            prfplcnm=hall.findtext('prfplcnm'),hall_seatscale_raw=hall.findtext('seatscale'),
            hall_seats=seats,hall_seat_status=status,xml_position=position,
            raw_source_path=row.path,raw_source_sha256=row.sha256))
halls=pd.DataFrame(hall_rows);facility_audit=pd.DataFrame(facility_rows)
halls['exact_key_occurrences']=halls.groupby(['mt10id','mt13id']).mt13id.transform('size')
halls['duplicate_exact_key']=halls.exact_key_occurrences.gt(1)
facility_audit['declared_xml_hall_count_match']=facility_audit.declared_hall_count.eq(facility_audit.xml_hall_count)
facility_audit['multiple_halls']=facility_audit.xml_hall_count.gt(1)|facility_audit.declared_hall_count.gt(1)
facility_audit['declared_and_observed_single_hall']=facility_audit.xml_hall_count.eq(1)&facility_audit.declared_hall_count.eq(1)
assert not facility_audit.mt10id.duplicated().any()
display(facility_audit[['xml_hall_count','facility_seats','multiple_halls','declared_xml_hall_count_match']].describe(include='all'))
display(halls[['mt10id','mt13id','prfplcnm','hall_seats','hall_seat_status']].head(8))

In [ ]:
flattened=facility_audit.merge(facilities[['mt10id','seatscale','mt13cnt']],on='mt10id',how='left',validate='one_to_one')
flattened['csv_seats']=flattened.seatscale.map(lambda value:parse_seat_integer(value)[0])
assert np.allclose(flattened.facility_seats,flattened.csv_seats,equal_nan=True,rtol=0,atol=0)
unique_halls=halls.loc[halls.exact_key_occurrences.eq(1)&halls.mt13id.ne('')].copy()
seat_link=performances[['mt20id','mt10id','mt13id','prfnm','prfpdfrom','prfpdto']].merge(
    unique_halls[['mt10id','mt13id','prfplcnm','hall_seats','hall_seat_status']],on=['mt10id','mt13id'],how='left',validate='many_to_one')
seat_link=seat_link.merge(facility_audit.drop(columns=['raw_source_path','raw_source_sha256']),on='mt10id',how='left',validate='many_to_one')
seat_link['in_study_candidate_pool']=seat_link.mt20id.isin(candidate_ids)
seat_link['hall_id_nonempty']=seat_link.mt13id.str.strip().ne('')
seat_link['exact_hall_id_match']=seat_link.hall_seat_status.notna()
seat_link['current_hall_seat_available']=seat_link.exact_hall_id_match & seat_link.hall_seat_status.eq('positive_integer')
seat_link['single_hall_current_seat_available']=seat_link.current_hall_seat_available & seat_link.declared_and_observed_single_hall
seat_link['multiple_hall_current_seat_available']=seat_link.current_hall_seat_available & seat_link.multiple_halls
seat_link['facility_to_hall_seat_ratio']=seat_link.facility_seats/seat_link.hall_seats.where(seat_link.current_hall_seat_available)
seat_link['facility_total_exceeds_matched_hall']=seat_link.current_hall_seat_available & seat_link.facility_seats.gt(seat_link.hall_seats)
seat_link['facility_total_less_than_matched_hall']=seat_link.current_hall_seat_available & seat_link.facility_seats.gt(0)&seat_link.facility_seats.lt(seat_link.hall_seats)
seat_link['historical_seats_verified']=False
assert not seat_link.mt20id.duplicated().any()
# 이름으로 보정하지 않는다. 원문 공연ID·시설ID·공연장ID 두 사례만 교차 확인한다.
raw_confirmation=[]
for multi in [False,True]:
    sample=seat_link.loc[seat_link.in_study_candidate_pool & seat_link.current_hall_seat_available & seat_link.multiple_halls.eq(multi)].iloc[0]
    record=raw.loc[raw.table.eq('performance_details')&raw.record_id.eq(sample.mt20id)].iloc[0]
    tree=ET.fromstring(register(record.path,record.sha256).read_bytes());db=tree.find('db')
    values={field:db.findtext(field) for field in ['mt20id','mt10id','mt13id']}
    assert all(values[field]==sample[field] for field in values)
    raw_confirmation.append({**values,'raw_source_path':record.path,'matched_csv':True})
display(pd.DataFrame(raw_confirmation))

**해석:** 정확한 연결의 단위는 `(mt10id, mt13id)`다. 이름이 같거나 시설에 공연장이 하나라는 추정만으로 누락된 ID를 채우지 않는다. 단일 공연장 시설은 복합시설의 전체 좌석 중복 위험이 작지만 현재 좌석이 과거에도 같았다는 검증은 여전히 없다.

## 3. 연결률과 시설 전체 좌석을 곱할 때의 위험
공연 ID와 상권에서 반복된 ID×분기를 구분해 분모를 기록한다. 개별 공연장 좌석이 있더라도 실제 관객·점유율·판매 가능한 좌석·과거 배치와 같다고 보지 않는다. 시설전체/개별공연장 좌석의 비율은 잘못된 곱셈의 산술 위험을 보여줄 뿐 관객 과대추정률을 추정한 결과가 아니다.

In [ ]:
coverage_rows=[]
for scope,part in [('all_fixed_performance_details',seat_link),('study_candidate_performance_IDs',seat_link.loc[seat_link.in_study_candidate_pool])]:
    for metric in ['hall_id_nonempty','exact_hall_id_match','current_hall_seat_available',
        'single_hall_current_seat_available','multiple_hall_current_seat_available',
        'facility_total_exceeds_matched_hall','facility_total_less_than_matched_hall']:
        numerator=int(part[metric].sum())
        coverage_rows.append(dict(scope=scope,metric=metric,numerator=numerator,denominator=len(part),pct=100*numerator/len(part)))
coverage=pd.DataFrame(coverage_rows)
pq_seats=pq.merge(seat_link[['mt20id','mt10id','mt13id','current_hall_seat_available','hall_seats','multiple_halls','facility_seats','facility_to_hall_seat_ratio']],on=['mt20id','mt10id'],how='left',validate='many_to_one')
pq_seats['joint_start_and_seat_eligible']=pq_seats.eligible_id_quarter & pq_seats.current_hall_seat_available
scheduled_total=int(pq_seats.known_scheduled_start_count.sum())
seat_scheduled=int(pq_seats.loc[pq_seats.joint_start_and_seat_eligible,'known_scheduled_start_count'].sum())
scheduled_coverage={'candidate_ID_quarters':len(pq_seats),'start_eligible_ID_quarters':int(pq_seats.eligible_id_quarter.sum()),
    'joint_start_and_current_seat_ID_quarters':int(pq_seats.joint_start_and_seat_eligible.sum()),
    'known_scheduled_starts_total':scheduled_total,'known_starts_with_current_hall_seats':seat_scheduled,
    'known_start_coverage_pct':100*seat_scheduled/scheduled_total}
risk=seat_link.loc[seat_link.in_study_candidate_pool & seat_link.multiple_halls & seat_link.current_hall_seat_available & seat_link.facility_seats.gt(0)]
risk_summary={'multi_hall_candidate_IDs_with_both_positive_seat_fields':len(risk),
    'facility_total_larger_than_hall_IDs':int(risk.facility_total_exceeds_matched_hall.sum()),
    'median_facility_to_hall_seat_ratio':float(risk.facility_to_hall_seat_ratio.median()),
    'max_facility_to_hall_seat_ratio':float(risk.facility_to_hall_seat_ratio.max())}
display(coverage)
display(pd.DataFrame([scheduled_coverage]))
display(risk.sort_values('facility_to_hall_seat_ratio',ascending=False)[['mt20id','mt10id','mt13id','prfnm','prfplcnm','facility_seats','hall_seats','facility_to_hall_seat_ratio']].head(8))

In [ ]:
hall_context=halls.merge(facility_audit[['mt10id','fcltynm','xml_hall_count','facility_seats']],on='mt10id',validate='many_to_one')
study_hall_keys=seat_link.loc[seat_link.in_study_candidate_pool,['mt10id','mt13id']].drop_duplicates()
matched_study_halls=hall_context.merge(study_hall_keys,on=['mt10id','mt13id'],validate='many_to_one')
seat_quality={'facility_XMLs_read':len(facility_audit),'hall_rows':len(halls),
    'duplicate_exact_hall_key_rows':int(halls.duplicate_exact_key.sum()),
    'declared_hall_count_mismatches':int((~facility_audit.declared_xml_hall_count_match).sum()),
    'hall_seat_status_counts':halls.hall_seat_status.value_counts().to_dict(),
    'matched_study_hall_max_seats':float(matched_study_halls.hall_seats.max()),
    'matched_study_hall_rows_over10000':int(matched_study_halls.hall_seats.gt(10000).sum()),
    'current_facility_csv_matches_XML_total':True,
    'historical_hall_seat_versions_observed':False,'audience_or_occupancy_observed':False}
display(matched_study_halls.sort_values('hall_seats',ascending=False)[['mt10id','mt13id','fcltynm','prfplcnm','hall_seats','facility_seats']].head(12))

**품질 해석:** 큰 좌석 수는 자동 오류로 간주하거나 임의 상한으로 자르지 않는다. 원문과 ID가 맞더라도 실제 좌석 배치·과거 좌석·관객이 확인된 것은 아니다. 중복 ID·비정형 좌석·현재 시설전체와 개별 좌석의 불일치를 별도 기록한다.

## 4. 현재 좌석 조건을 추가할 때의 상권·분기 가용률
파싱 전 전체 공식 후보 ID를 분모로 유지하고, 시작일정과 양의 정수 개별 공연장 좌석을 함께 갖춘 ID를 분자로 센다. 누락 좌석을 0석으로 바꾸지 않는다. 후보 없는 0은 기존 고정 등록 범위 안의 0이다.

In [ ]:
relations=links[['area_id','spatial_definition','mt10id']].merge(pq_seats,on='mt10id',how='inner',validate='many_to_many')
seat_aq=relations.groupby(['area_id','quarter','spatial_definition']).agg(candidate_id_count=('mt20id','nunique'),
    start_eligible_id_count=('eligible_id_quarter','sum'),seat_available_id_count=('current_hall_seat_available','sum'),
    joint_eligible_id_count=('joint_start_and_seat_eligible','sum')).reset_index()
seat_aq=exposure[['area_id','quarter','spatial_definition','candidate_id_count','availability_pct']].merge(
    seat_aq,on=['area_id','quarter','spatial_definition'],how='left',suffixes=('_source',''),validate='one_to_one')
for column in ['candidate_id_count','start_eligible_id_count','seat_available_id_count','joint_eligible_id_count']:
    seat_aq[column]=seat_aq[column].fillna(0).astype(int)
assert seat_aq.candidate_id_count.eq(seat_aq.candidate_id_count_source).all()
seat_aq['joint_availability_pct']=100*seat_aq.joint_eligible_id_count/seat_aq.candidate_id_count.replace(0,np.nan)
seat_aq.loc[seat_aq.candidate_id_count.eq(0),'joint_availability_pct']=100
seat_aq['joint_complete']=seat_aq.joint_eligible_id_count.eq(seat_aq.candidate_id_count)
assert seat_aq.joint_availability_pct.le(seat_aq.availability_pct+1e-10).all()
seat_sample_rows=[]
common=panel.total_sales.gt(0)&panel.night_sales.notna()&panel.night_sales.ge(0)
for space in ['centroid500','min_distance500']:
    for threshold in [100,90]:
        chosen=seat_aq.loc[seat_aq.spatial_definition.eq(space)&seat_aq.joint_availability_pct.ge(threshold)]
        joined=panel.loc[common].merge(chosen[['area_id','quarter']],on=['area_id','quarter'],validate='many_to_one')
        repetitions=joined.groupby(['area_id','industry_id']).quarter.transform('nunique')
        repeated=joined.loc[repetitions.ge(2)]
        seat_sample_rows.append(dict(spatial_definition=space,joint_threshold_pct=threshold,joint_area_quarters=len(chosen),
            common_outcome_rows=len(joined),common_outcome_areas=int(joined.area_id.nunique()),
            repeated_entity_rows=len(repeated),repeated_entity_areas=int(repeated.area_id.nunique())))
seat_samples=pd.DataFrame(seat_sample_rows)
display(seat_samples)

## 5. 핵심 상권에서 어떤 후보가 제외되는가
중심점 500m 후보에 대해 12번의 가용 판정을 그대로 사용한다. 중복 합산을 막기 위해 대표 제외 사유는 기간 무효→일정 파싱 실패→정상 러닝타임 자체 겹침→해당 분기 일반요일 시작 없음 순서로 정한다. 실제 일정 검증 우선 목록이지 파서 수정이나 제외 완화의 근거를 자동으로 만드는 목록은 아니다.

In [ ]:
core_ids=['3120012','3120103']
core=relations.loc[relations.spatial_definition.eq('centroid500') & relations.area_id.isin(core_ids)].copy()
core=core.merge(areas[['area_id','area_name','district']],on='area_id',validate='many_to_one')
core=core.merge(audit[['mt20id','dtguidance','prfruntime','prfpdfrom','prfpdto','own_runtime_overlap_ambiguous']],on='mt20id',validate='many_to_one')
core=core.merge(performances[['mt20id','prfnm']],on='mt20id',validate='many_to_one')
core['primary_exclusion_reason']=np.select([
    ~core.valid_period,core.schedule_status.ne('ok'),core.excluded_known_valid_runtime_overlap,
    core.no_matching_regular_start],[
    'invalid_registered_period','schedule:'+core.schedule_status.astype(str),
    'overlap_with_valid_runtime','no_regular_start_in_source_quarter'],default='eligible')
assert core.primary_exclusion_reason.eq('eligible').equals(core.eligible_id_quarter)
assert not core.duplicated(['area_id','quarter','mt20id']).any()
core_quarters=core.groupby(['area_id','area_name','quarter']).agg(candidate_IDs=('mt20id','nunique'),
    eligible_IDs=('eligible_id_quarter','sum'),joint_seat_eligible_IDs=('joint_start_and_seat_eligible','sum')).reset_index()
core_quarters['excluded_IDs']=core_quarters.candidate_IDs-core_quarters.eligible_IDs
core_quarters['start_availability_pct']=100*core_quarters.eligible_IDs/core_quarters.candidate_IDs
core_quarters['joint_availability_pct']=100*core_quarters.joint_seat_eligible_IDs/core_quarters.candidate_IDs
excluded=core.loc[~core.eligible_id_quarter].copy()
core_causes=excluded.groupby(['area_id','area_name','primary_exclusion_reason']).agg(
    excluded_ID_quarters=('mt20id','size'),unique_excluded_IDs=('mt20id','nunique'),affected_quarters=('quarter','nunique')).reset_index()
core_total=core.groupby(['area_id','area_name']).agg(candidate_ID_quarters=('mt20id','size'),unique_candidate_IDs=('mt20id','nunique'),
    eligible_ID_quarters=('eligible_id_quarter','sum'),joint_seat_eligible_ID_quarters=('joint_start_and_seat_eligible','sum')).reset_index()
core_total['excluded_ID_quarters']=core_total.candidate_ID_quarters-core_total.eligible_ID_quarters
core_causes=core_causes.merge(core_total[['area_id','excluded_ID_quarters']],on='area_id',validate='many_to_one')
core_causes['share_of_excluded_ID_quarters_pct']=100*core_causes.excluded_ID_quarters_x/core_causes.excluded_ID_quarters_y
core_causes=core_causes.rename(columns={'excluded_ID_quarters_x':'excluded_ID_quarters','excluded_ID_quarters_y':'all_excluded_ID_quarters'})
display(core_total)
display(core_causes.sort_values(['area_id','excluded_ID_quarters'],ascending=[True,False]))

In [ ]:
priority=excluded.groupby(['area_id','area_name','mt20id','mt10id','mt13id','prfnm'],dropna=False).agg(
    affected_source_quarters=('quarter',lambda x:','.join(sorted(x.unique()))),affected_quarter_count=('quarter','nunique'),
    reasons=('primary_exclusion_reason',lambda x:';'.join(sorted(x.unique()))),
    dtguidance=('dtguidance','first'),prfruntime=('prfruntime','first'),prfpdfrom=('prfpdfrom','first'),prfpdto=('prfpdto','first'),
    current_hall_seat_available=('current_hall_seat_available','first'),current_hall_seats=('hall_seats','first'),
    runtime_overlap_flag=('own_runtime_overlap_ambiguous','first')).reset_index()
priority=priority.sort_values(['affected_quarter_count','area_id','mt20id'],ascending=[False,True,True]).reset_index(drop=True)
priority['priority_order']=np.arange(1,len(priority)+1)
priority['recommended_check']='verify historical date-specific starts, exceptions, duration and hall ID using original records; no automatic parser relaxation'
display(priority[['priority_order','area_name','mt20id','prfnm','affected_quarter_count','reasons','dtguidance','prfruntime']].head(12))

**해석:** 같은 ID가 여러 분기에 문제를 만드는 경우를 먼저 볼 수 있게 정렬했다. 현재 원문에 적힌 일반요일 일정과 러닝타임이 겹친다는 플래그는 실제 오류·휴연을 확인한 것이 아니다. 티켓 입장시간, 복수 운영 방식, 과거 일정 병합 가능성 등은 별도 원문 검증이 필요하며 이 감사에서는 임의로 수정하지 않는다.

## 6. 저장과 현재 좌석의 활용 범위
공식 개별 ID 연결이 가능하더라도 현재 좌석 규모를 사용한 예정 공급의 대리지표일 뿐이다. 과거 좌석과 실제 관객·점유율을 검증한 결과는 아니다. 이 노트북에서는 회귀를 수행하지 않는다.

In [ ]:
data_outputs={'06_hall_catalog.csv':halls,'06_facility_seat_audit.csv':facility_audit,
    '06_performance_seat_link_audit.csv':seat_link,'06_performance_quarter_seat_audit.csv':pq_seats,
    '06_seat_coverage_area_quarter.csv':seat_aq,'06_core_candidate_exclusion_audit.csv':core}
report_outputs={'06_seat_coverage.csv':coverage,'06_seat_sample_availability.csv':seat_samples,
    '06_core_quarter_coverage.csv':core_quarters,'06_core_exclusion_causes.csv':core_causes,
    '06_core_verification_priority.csv':priority,'06_core_summary.csv':core_total,
    '06_raw_ID_confirmation.csv':pd.DataFrame(raw_confirmation)}
outputs={}
for directory,mapping in [(DATA,data_outputs),(REPORTS,report_outputs)]:
    for name,frame in mapping.items():
        path=directory/name;frame.to_csv(path,index=False,encoding='utf-8-sig')
        outputs[path.relative_to(ROOT).as_posix()]={'rows':len(frame),'columns':frame.columns.tolist(),'sha256':sha(path)}
for relative,record in inputs.items():
    record['sha256_after']=sha(project_path(relative));record['unchanged']=record['sha256_before']==record['sha256_after']
assert all(record['unchanged'] for record in inputs.values())
def finite(value):
    if isinstance(value,dict):return {k:finite(v) for k,v in value.items()}
    if isinstance(value,list):return [finite(v) for v in value]
    if isinstance(value,(float,np.floating)) and not np.isfinite(value):return None
    if isinstance(value,np.generic):return value.item()
    return value
summary=finite({'status':'complete','created_utc':pd.Timestamp.now(tz='UTC').isoformat(),
    'role':'current_hall_seat_linkability_and_core_candidate_audit','new_regression_fits':0,'network_requests':0,
    'all_inputs_unchanged':True,'kopis_fixed_snapshot_run_id':manifest['run_id'],
    'individual_hall_array_present_in_raw_XML':True,'individual_hall_array_in_flat_facility_CSV':False,
    'observed_hall_fields':sorted(hall_tags),'official_link_key':['mt10id','mt13id'],
    'facility_name_inference_used':False,'missing_seats_filled_with_zero':False,
    'current_seat_link_definition':'unique same-facility exact mt13id match with positive integer individual hall seatscale',
    'historical_seats_verified':False,'actual_audience_observed':False,
    'seat_quality':seat_quality,'performance_ID_coverage':coverage.to_dict('records'),
    'known_scheduled_start_coverage':scheduled_coverage,'facility_total_multiplication_risk':risk_summary,
    'seat_sample_availability':seat_samples.to_dict('records'),'core_summary':core_total.to_dict('records'),
    'core_exclusion_causes':core_causes.to_dict('records'),
    'priority_list_rows':len(priority),'inputs':list(inputs.values()),'outputs':outputs,
    'limitations':['snapshot seat values and hall assignment are current, not verified historic capacities',
        'positive integer seat value is a structural validity check, not a factual capacity validation',
        'facility total seat count must not be assigned to every hall performance',
        'known starts and seats describe a scheduled current-capacity proxy, not audience or actual seat supply',
        'core exclusions are preexisting strict schedule rules; this audit does not validate or change them']})
(REPORTS/'06_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
print(f'완료: 개별공연장 {len(halls):,}행, 공연ID 연결감사 {len(seat_link):,}행, 핵심상권 검증목록 {len(priority):,}행; 입력 {len(inputs)}개 불변')